In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ============================================
# 1. CHARGER LES DONNÉES
# ============================================
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# ============================================
# 2. NETTOYAGE BOURRIN - TRAIN
# ============================================
df_train_clean = df_train.dropna(axis=1)
df_train_clean = df_train_clean.select_dtypes(include=['int64', 'float64', 'bool'])
df_train_clean = df_train_clean.loc[:, df_train_clean.nunique() > 1]

# ============================================
# 3. SÉPARATION X / y - TRAIN
# ============================================
X_train = df_train_clean.drop('is_fraud', axis=1)
y_train = df_train_clean['is_fraud']

# ============================================
# 4. NETTOYAGE BOURRIN - TEST
# ============================================
colonnes_a_garder = X_train.columns.tolist()
df_test_clean = df_test[colonnes_a_garder + ['customer_id']].copy()
df_test_clean = df_test_clean.dropna()
customer_ids = df_test_clean['customer_id']
X_test = df_test_clean[colonnes_a_garder]

# ============================================
# 5. MODEL - DECISION TREE
# ============================================
model = DecisionTreeClassifier(max_depth=10, random_state=42)

# Cross-validation sur le train
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall')

# Entraînement
model.fit(X_train, y_train)

# Prédictions sur train (pour évaluation)
y_train_pred = model.predict(X_train)

# Prédictions sur test
y_pred = model.predict(X_test)

# ============================================
# 6. ÉVALUATION SUR TRAIN
# ============================================
print(f"CV Recall: {cv_scores.mean():.4f}")
print(f"Accuracy:  {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Precision: {precision_score(y_train, y_train_pred):.4f}")
print(f"Recall:    {recall_score(y_train, y_train_pred):.4f}")
print(f"F1-Score:  {f1_score(y_train, y_train_pred):.4f}")
print(confusion_matrix(y_train, y_train_pred))

# ============================================
# 7. PRÉDICTIONS TEST
# ============================================
predictions_df = pd.DataFrame({
    'customer_id': customer_ids,
    'target': y_pred
})

print(predictions_df.to_string(index=False))